# 💳 Project Task: GoPay Fintech Analytics
## Data Cleaning, Feature Engineering & Exploratory Data Analysis

**Module 2 — Python for Data Analysis | Purwadhika Digital Technology School**

---

> ⚠️ **Jangan di-run dulu.** Copy notebook ini terlebih dahulu, baru kerjakan di file copy-an kamu.

---

### Konteks Bisnis
GoPay telah berkembang menjadi tulang punggung ekosistem Super App GoTo. Fitur GoPayLater — layanan kredit berbasis limit — berhasil mendongkrak GTV, namun kini menghadapi masalah serius: tingkat NPL (Non-Performing Loan) yang meningkat, bug validasi limit kredit, dan inkonsistensi data dari puluhan micro-service.

Kamu berperan sebagai Data Analyst di tim **Risk Management GoPay** yang diminta untuk membersihkan data, mengidentifikasi pola gagal bayar, dan memberikan rekomendasi perbaikan credit scoring.

**Dataset (3 tabel):**
- `gopay_users.csv` — 35.000 baris (Dimensi User)
- `gopay_services.csv` — 20 baris (Dimensi Layanan)
- `gopay_transactions.csv` — 300.000 baris (Fakta Transaksi)

---

### ⚠️ Catatan Penting
- Task ini **open-ended** — tidak ada satu jawaban yang mutlak benar
- Yang dinilai: **ketepatan keputusan**, **kualitas justifikasi**, dan **kedalaman analisis**
- Setiap keputusan di Data Cleaning & Feature Engineering **wajib disertai penjelasan** di markdown cell
- EDA dikerjakan **tanpa visualisasi** — gunakan pandas aggregation, filtering, sorting, dan merge

---

### 🚨 Business Context Error (Wajib Diinvestigasi)
> Lebih dari **50% transaksi GoPayLater** memiliki `amount` yang **melebihi `paylater_limit`** user yang bersangkutan.  
> Ini adalah bug sistematis pada validasi limit kredit — bukan sekadar outlier biasa.  
> Identifikasi, kuantifikasi dampak finansialnya, dan rekomendasikan perbaikan.

---
## 0. Import & Load Data

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

In [2]:
# Load semua dataset
# Sesuaikan path dengan lokasi file kamu
df_users_raw    = pd.read_csv('gopay_users.csv')
df_services_raw = pd.read_csv('gopay_services.csv')
df_trx_raw      = pd.read_csv('gopay_transactions.csv')

# Buat copy untuk dikerjakan
df_users    = df_users_raw.copy()
df_services = df_services_raw.copy()
df_trx      = df_trx_raw.copy()

print(f'users       : {df_users.shape}')
print(f'services    : {df_services.shape}')
print(f'transactions: {df_trx.shape}')

users       : (35000, 5)
services    : (20, 3)
transactions: (300000, 8)


---
## 2. Data Cleaning

### 2.1 Eksplorasi Awal (Wajib)

Lakukan eksplorasi menyeluruh pada **ketiga tabel** sebelum membersihkan data apapun.

In [3]:
# Shape dan info umum — lakukan untuk ketiga tabel
print(f"Shape dari tabel Users: {df_users.shape}")
df_users.info()
print()

Shape dari tabel Users: (35000, 5)
<class 'pandas.DataFrame'>
RangeIndex: 35000 entries, 0 to 34999
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   user_id                35000 non-null  str    
 1   join_date              35000 non-null  str    
 2   gopay_tier             35000 non-null  str    
 3   internal_credit_score  29750 non-null  float64
 4   paylater_limit         35000 non-null  int64  
dtypes: float64(1), int64(1), str(3)
memory usage: 1.3 MB



In [4]:

print(f"Shape dari tabel Services: {df_services.shape}")
df_services.info()
print()

Shape dari tabel Services: (20, 3)
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   service_id    20 non-null     str  
 1   service_name  20 non-null     str  
 2   category      20 non-null     str  
dtypes: str(3)
memory usage: 612.0 bytes



In [5]:
print(f"Shape dari tabel Transactions: {df_trx.shape}")
df_trx.info()
print()

Shape dari tabel Transactions: (300000, 8)
<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   trx_id          300000 non-null  str    
 1   user_id         300000 non-null  str    
 2   service_id      300000 non-null  str    
 3   trx_date        300000 non-null  str    
 4   payment_method  300000 non-null  str    
 5   amount          300000 non-null  int64  
 6   late_fee        300000 non-null  float64
 7   payment_status  300000 non-null  str    
dtypes: float64(1), int64(1), str(6)
memory usage: 18.3 MB



In [6]:
# Tipe data seluruh kolom
print(f"Tipe data dari tabel Users:")
df_users.dtypes

Tipe data dari tabel Users:


user_id                      str
join_date                    str
gopay_tier                   str
internal_credit_score    float64
paylater_limit             int64
dtype: object

In [7]:
print(f"Tipe data dari tabel Services:")
df_services.dtypes

Tipe data dari tabel Services:


service_id      str
service_name    str
category        str
dtype: object

In [8]:
print(f"Tipe data dari tabel Transactions:")
df_trx.dtypes

Tipe data dari tabel Transactions:


trx_id                str
user_id               str
service_id            str
trx_date              str
payment_method        str
amount              int64
late_fee          float64
payment_status        str
dtype: object

In [8]:
naUsers = df_users.isnull().sum()
naServices = df_services.isnull().sum()
naTrans = df_trx.isnull().sum()

display(naUsers, naServices, naTrans)

user_id                     0
join_date                   0
gopay_tier                  0
internal_credit_score    5250
paylater_limit              0
dtype: int64

service_id      0
service_name    0
category        0
dtype: int64

trx_id            0
user_id           0
service_id        0
trx_date          0
payment_method    0
amount            0
late_fee          0
payment_status    0
dtype: int64

In [9]:
# Missing values: jumlah dan persentase per kolom, per tabel
# Tampilkan hanya kolom yang memiliki missing values
naUsers = df_users.isnull().sum()
naServices = df_services.isnull().sum()
naTrans = df_trx.isnull().sum()

display(
    naUsers[naUsers > 0], 
    naServices[naServices > 0], 
    naTrans[naTrans > 0]
    )

internal_credit_score    5250
dtype: int64

Series([], dtype: int64)

Series([], dtype: int64)

In [10]:
# Distribusi kolom-kolom kritis
# payment_method, amount, late_fee, payment_status, gopay_tier, internal_credit_score
# in df users: gopay tier, internal credit
# in df trans: payment method, amount, late fee, payment status

dfMerge = df_users[["user_id", "gopay_tier", "internal_credit_score"]].merge(df_trx[["user_id", "payment_method", "amount", "late_fee", "payment_status"]],
on="user_id", how="left")

dfMerge

,user_id,gopay_tier,internal_credit_score,payment_method,amount,late_fee,payment_status
0,GP-000001,Basic,NaN,GoPay,704068.00,0.00,Paid
1,GP-000001,Basic,NaN,GO-PAY,195808.00,0.00,Paid
2,GP-000001,Basic,NaN,gopay,606859.00,0.00,Paid
3,GP-000001,Basic,NaN,GoPay,356899.00,0.00,Paid
4,GP-000001,Basic,NaN,GoPayLater,186928.00,0.00,Paid
...,...,...,...,...,...,...,...
300003,GP-035000,Basic,362.00,GoPay,236650.00,0.00,Paid
300004,GP-035000,Basic,362.00,gopay,138743.00,0.00,Paid
300005,GP-035000,Basic,362.00,GoPayLater,749400.00,0.00,Paid
300006,GP-035000,Basic,362.00,gopay,568217.00,0.00,Paid


In [11]:
dfMerge.loc[:, "payment_method"].value_counts()

payment_method
GoPay          90240
GoPayLater     60014
gopay          59854
PayLater       30017
GO-PAY         15136
Cash           14978
CASH           14972
gopay_later    14789
Name: count, dtype: int64

In [12]:
dfMerge.loc[:, "late_fee"].value_counts()

late_fee
0.00           284423
99999999.00        76
-15000.00          75
45948.00            5
22678.00            5
                ...  
49453.00            1
36958.00            1
8627.00             1
22789.00            1
10905.00            1
Name: count, Length: 13069, dtype: int64

In [13]:
# Cek konsistensi relasi antar tabel
# Apakah semua service_id di transactions ada di services?
# Apakah semua user_id di transactions ada di users?
df_trx["service_id"].isin(df_services["service_id"].unique()).sum()

np.int64(300000)

In [14]:
(df_trx["user_id"].isin(df_users["user_id"].unique())).sum()

np.int64(300000)

**✍️ Ringkasan Temuan Eksplorasi:**

*(Kolom apa yang bermasalah di setiap tabel, seberapa parah, dan prioritas penanganan kamu)*

Kami menemukan ada missing value di:
data user sebanyak 15%(5250), kolom: internal_credit_score

Kami juga menemukan inkonsistensi di:
df transaction:
kolom late_fee: ada nominal negative (-15000, 75 row, 999999 di 76 row)
kolom payment method: ada beragam penulisan GoPay, gopay, PayLater, GoPayLater, Cash, CASH yang perlu distandarisasi.

---
### 2.2 Kerangka Identifikasi Missing Values

Sebelum menangani missing values pada kolom manapun, identifikasi dulu **jenis missing value-nya**.

| Jenis | Definisi Singkat | Implikasi Penanganan | Contoh di Dataset Ini |
|---|---|---|---|
| **MCAR** *(Missing Completely At Random)* | Nilai kosong tidak berkaitan dengan kolom lain. Pola missing benar-benar acak. | Relatif aman di-impute atau di-drop tanpa bias signifikan. | Sebagian kecil `internal_credit_score` kosong karena gangguan sistem scraping acak. |
| **MAR** *(Missing At Random)* | Nilai kosong berkaitan dengan kolom **lain**, bukan dengan nilai kolom itu sendiri. | Imputation berbasis kolom lain lebih tepat. Drop bisa menyebabkan bias. | `internal_credit_score` kosong mungkin berkorelasi dengan `gopay_tier` atau `join_date`. |
| **MNAR** *(Missing Not At Random)* | Nilai kosong berkaitan langsung dengan nilai yang seharusnya ada. Ada alasan sistematis. | Imputation apapun berisiko misleading. Perlu keputusan bisnis eksplisit. | `internal_credit_score` kosong justru karena user tidak pernah bertransaksi — nilai kosong itu sendiri adalah sinyal risiko. |

> 💡 **Cara Menggunakan Kerangka Ini:**
> Untuk setiap kolom bermasalah, tanyakan:
> 1. Apakah pola missing-nya acak, atau ada pola tertentu?
> 2. Apakah nilai kosong berkaitan dengan kolom lain?
> 3. Apakah nilai kosong itu sendiri mengandung informasi bisnis?
>
> Justifikasi reasoning kamu lebih penting dari labelnya.

In [15]:
dfMerge.loc[:, dfMerge.notnull().any(axis = 0)]

,user_id,gopay_tier,internal_credit_score,payment_method,amount,late_fee,payment_status
0,GP-000001,Basic,NaN,GoPay,704068.00,0.00,Paid
1,GP-000001,Basic,NaN,GO-PAY,195808.00,0.00,Paid
2,GP-000001,Basic,NaN,gopay,606859.00,0.00,Paid
3,GP-000001,Basic,NaN,GoPay,356899.00,0.00,Paid
4,GP-000001,Basic,NaN,GoPayLater,186928.00,0.00,Paid
...,...,...,...,...,...,...,...
300003,GP-035000,Basic,362.00,GoPay,236650.00,0.00,Paid
300004,GP-035000,Basic,362.00,gopay,138743.00,0.00,Paid
300005,GP-035000,Basic,362.00,GoPayLater,749400.00,0.00,Paid
300006,GP-035000,Basic,362.00,gopay,568217.00,0.00,Paid


---
### 2.3 Penanganan Kolom `payment_method`

Kolom ini memiliki 8 varian penulisan untuk 3 metode pembayaran yang berbeda, akibat inkonsistensi penamaan antar micro-service.

| Varian Asli | Metode Sebenarnya |
|---|---|
| `GoPay`, `gopay`, `GO-PAY` | GoPay (saldo digital) |
| `GoPayLater`, `PayLater`, `gopay_later` | GoPayLater (kredit) |
| `Cash`, `CASH` | Cash |

> 🧠 **Critical Thinking Prompt:**  
> Setelah standarisasi, periksa ulang: apakah ada user **Basic tier** yang menggunakan GoPayLater?  
> Secara aturan bisnis, PayLater hanya boleh digunakan oleh user Plus.  
> Jika ada, apakah itu error data atau bug sistem validasi?

In [16]:
# Lihat semua nilai unik di payment_method beserta frekuensinya
dfMerge["payment_method"].unique()

<StringArray>
[      'GoPay',      'GO-PAY',       'gopay',  'GoPayLater', 'gopay_later',
    'PayLater',        'Cash',        'CASH',           nan]
Length: 9, dtype: str

**✍️ Mapping standarisasi yang kamu buat:**
- Varian asli → nilai standar (tuliskan mapping lengkapnya):
- Format standar yang kamu pilih dan alasannya:
- Temuan setelah standarisasi (apakah ada Basic tier yang pakai GoPayLater?):

> 

In [17]:
# TODO: Standarisasi payment_method
# Simpan hasil ke kolom baru: payment_method_clean

dfMerge["payment_method_clean"] = dfMerge["payment_method"].replace({
    "gopay": "GoPay",
    "GO-PAY": "GoPay",
    "GoPayLater": "PayLater",
    "gopay_later": "PayLater",
    "CASH": "Cash"
})

dfMerge.drop(columns="payment_method")
display(dfMerge)

,user_id,gopay_tier,internal_credit_score,payment_method,amount,late_fee,payment_status,payment_method_clean
0,GP-000001,Basic,NaN,GoPay,704068.00,0.00,Paid,GoPay
1,GP-000001,Basic,NaN,GO-PAY,195808.00,0.00,Paid,GoPay
2,GP-000001,Basic,NaN,gopay,606859.00,0.00,Paid,GoPay
3,GP-000001,Basic,NaN,GoPay,356899.00,0.00,Paid,GoPay
4,GP-000001,Basic,NaN,GoPayLater,186928.00,0.00,Paid,PayLater
...,...,...,...,...,...,...,...,...
300003,GP-035000,Basic,362.00,GoPay,236650.00,0.00,Paid,GoPay
300004,GP-035000,Basic,362.00,gopay,138743.00,0.00,Paid,GoPay
300005,GP-035000,Basic,362.00,GoPayLater,749400.00,0.00,Paid,PayLater
300006,GP-035000,Basic,362.00,gopay,568217.00,0.00,Paid,GoPay


In [ ]:
# Verifikasi: cek Basic tier yang menggunakan GoPayLater setelah standarisasi


---
### 2.4 Penanganan `internal_credit_score`

Kolom `internal_credit_score` di tabel users memiliki **5.250 nilai kosong (~15%)** — variabel kritis untuk analisis risiko kredit.

> 🧠 **Critical Thinking Prompt:**  
> Apakah nilai kosong ini karena sistem gagal mencatat, atau karena user memang belum punya histori kredit?  
> User tanpa credit score = *unscored* — di industri fintech, ini dianggap risiko tersendiri.  
> Keputusan kamu di sini akan langsung mempengaruhi hasil analisis profil risiko di Section 4.

In [18]:
# Investigasi pola missing values pada internal_credit_score
# Apakah berkorelasi dengan gopay_tier, paylater_limit, atau join_date?

# Display tabel dimana rows adalah semua df_users yang memiliki credit score kosong, dan berkolom gopay_tier, paylater_limit, atau join_date
# Karena tier basic sudah dipastikan akan punya paylater limit = 0, maka untuk instance ini diexclude saja.
result = df_users.loc[
    df_users["internal_credit_score"].isnull() & (df_users["gopay_tier"] != "Basic"),
    ["gopay_tier", "paylater_limit", "join_date"]
]
result.head(15)


# Basic -> paylaterlimit = 0
# plus -> paylaterlimit = 500.000, 5.000.000, 3.000.000, 500.000, 1.500.000

,gopay_tier,paylater_limit,join_date
7,Plus,500000,2023-03-26
9,Plus,5000000,2021-04-20
19,Plus,3000000,2022-06-23
25,Plus,500000,2022-01-15
35,Plus,1500000,2022-07-20
39,Plus,5000000,2023-04-27
77,Plus,1500000,2021-03-22
102,Plus,3000000,2022-07-13
116,Plus,1500000,2023-04-19
119,Plus,500000,2022-01-27


In [19]:
# Cek: apakah user yang credit_score-nya missing lebih banyak yang default?
# Hint: merge dengan df_trx, lalu bandingkan default rate

#Buat dataframe baru yang merupakan merge dari users dan transaction. Mirip dengan dfMerge namun hanya menggabungkan internal_credit_score dari df_users
dfUserTrx = df_users[["user_id", "internal_credit_score"]].merge(df_trx,
on="user_id", how="left")

In [20]:
# Unique user counts
null_score_users = dfUserTrx[dfUserTrx['internal_credit_score'].isna()]['user_id'].nunique()
default_null_score_users = dfUserTrx[
    (dfUserTrx['internal_credit_score'].isna()) & 
    (dfUserTrx['payment_status'] == 'Default')
]['user_id'].nunique()

print(f"Users with Null Credit Score: {null_score_users:,}")
print(f"Users with Null Credit Score & Default: {default_null_score_users:,}")
print(f"Default Rate among Null Users: {default_null_score_users / null_score_users:.2%}")

Users with Null Credit Score: 5,250
Users with Null Credit Score & Default: 1,860
Default Rate among Null Users: 35.43%


**✍️ Analisis & Justifikasi:**
- **Jenis missing value (MCAR / MAR / MNAR):** dan alasan klasifikasi kamu:
- Temuan investigasi pola missing (berkorelasi dengan tier? join_date?):
- Apakah user tanpa credit score memiliki default rate yang berbeda?
- Keputusan penanganan (drop / impute / pertahankan NaN) dan alasan:

> 

***Penganan internal_credit_score = NaN***

Kolom `internal_credit_score` di tabel users memiliki **5.250 nilai kosong (~15%)** — variabel kritis untuk analisis risiko kredit.

**Jenis missing value**: MCAR.

**Pola missing**: 
Tidak ditemukan korelasi apapun dengan kolom join date, gopay tier, maupun paylater limit.

**Apakah user tanpa credit score memiliki default rate yang beda**:
Ya, berbeda. Namun Default rate dalam user tanpa credit score adalah 35.43%, sehingga tidak terlalu bisa dikorelasikan dengan ketidakadaan nilai internal_credit_score

**Keputusan akhir**:
Di-impute menjadi 0. Kami tidak berencana untuk dropping karena dengan itu kita akan kehilangan 15% data

In [30]:
(df_users["internal_credit_score"] == 0).sum()

np.int64(0)

In [21]:
#Making sure bahwa semua yang ber-tier Basic pasti tidak punya paylater dan internal_credit_score

dfBasicNan = dfMerge.loc[
    dfMerge["internal_credit_score"].isnull() & (dfMerge["gopay_tier"] == "Basic")
]
dfBasicNan

,user_id,gopay_tier,internal_credit_score,payment_method,amount,late_fee,payment_status,payment_method_clean
0,GP-000001,Basic,NaN,GoPay,704068.00,0.00,Paid,GoPay
1,GP-000001,Basic,NaN,GO-PAY,195808.00,0.00,Paid,GoPay
2,GP-000001,Basic,NaN,gopay,606859.00,0.00,Paid,GoPay
3,GP-000001,Basic,NaN,GoPay,356899.00,0.00,Paid,GoPay
4,GP-000001,Basic,NaN,GoPayLater,186928.00,0.00,Paid,PayLater
...,...,...,...,...,...,...,...,...
299923,GP-034992,Basic,NaN,gopay,529392.00,0.00,Paid,GoPay
299924,GP-034992,Basic,NaN,GoPay,70341.00,0.00,Paid,GoPay
299925,GP-034992,Basic,NaN,GoPay,286566.00,0.00,Paid,GoPay
299926,GP-034992,Basic,NaN,GoPayLater,389409.00,0.00,Paid,PayLater


In [22]:
(dfBasicNan["gopay_tier"] == "Basic").sum()

# All with Basic tiers will have

np.int64(18222)

In [28]:
dfMerge.head(5)

,user_id,gopay_tier,internal_credit_score,payment_method,amount,late_fee,payment_status,payment_method_clean
6,GP-000002,Plus,751.00,gopay,200782.00,0.00,Paid,GoPay
7,GP-000002,Plus,751.00,GoPay,251279.00,0.00,Paid,GoPay
8,GP-000002,Plus,751.00,GoPay,191460.00,0.00,Paid,GoPay
9,GP-000002,Plus,751.00,gopay,88206.00,0.00,Paid,GoPay
10,GP-000002,Plus,751.00,GoPayLater,357436.00,37815.00,Default,PayLater


In [44]:
df_users.loc[(df_users["gopay_tier"] == "Basic") & (df_users["paylater_limit"] != 0)]

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit


---
### 2.5 Penanganan `late_fee`

Kolom `late_fee` memiliki dua jenis anomali yang berbeda sifatnya — tangani secara terpisah.

> 🧠 **Critical Thinking Prompt:**  
> Di industri fintech, denda keterlambatan diatur oleh regulasi OJK.  
> Nilai `late_fee` yang sangat besar bisa berarti bug sistem, bukan kebijakan yang valid.  
> Keputusan kamu harus mempertimbangkan aspek **compliance**, bukan hanya statistik.

In [ ]:
# Investigasi distribusi late_fee secara menyeluruh
# Berapa nilai negatif? Berapa nilai ekstrem?


In [ ]:
# Anomali 1: Nilai negatif
# Apakah terjadi pada payment_status tertentu?


In [ ]:
# Anomali 2: Nilai ekstrem tinggi (> Rp 10 juta)
# Apakah ada pola pada service atau user tertentu?


**✍️ Analisis & Justifikasi — Anomali 1 (late_fee negatif):**
- Jumlah baris terdampak:
- Hipotesis penyebab (logical error? refund denda yang salah catat?):
- Keputusan penanganan dan alasan:

> 

**✍️ Analisis & Justifikasi — Anomali 2 (late_fee ekstrem):**
- Jumlah baris terdampak dan range nilainya:
- Threshold yang kamu pilih untuk mendefinisikan 'ekstrem' dan alasannya:
- Hipotesis penyebab (bug sistem? kebijakan tidak terkontrol?):
- Keputusan penanganan (cap / drop / flag) dan alasan:

> 

In [ ]:
# TODO: Implementasi penanganan Anomali 1 (late_fee negatif)


In [ ]:
# TODO: Implementasi penanganan Anomali 2 (late_fee ekstrem)


---
### 2.6 Penanganan Anomali Tanggal: `trx_date` sebelum `join_date`

Terdapat **~30.177 transaksi (~10%)** dengan `trx_date` lebih awal dari `join_date` user — secara logika bisnis tidak mungkin terjadi.

> 🧠 **Critical Thinking Prompt:**  
> Di konteks fintech, transaksi sebelum akun dibuat bisa mengindikasikan **fraud** atau **data migration issue**.  
> Drop vs. flag memiliki implikasi berbeda: drop menghilangkan sinyal fraud, flag mempertahankannya untuk analisis.  
> Apakah anomali ini lebih banyak terjadi pada user yang akhirnya **Default**?

In [ ]:
# Konversi kolom tanggal ke datetime


In [ ]:
# Identifikasi transaksi dengan trx_date < join_date
# Investigasi: seberapa besar selisih tanggalnya? Distribusi selisih negatif?


In [ ]:
# Apakah anomali ini berkorelasi dengan payment_status = Default?
# Apakah tersebar merata atau terkonsentrasi pada user/tanggal tertentu?


**✍️ Analisis & Justifikasi:**
- Jumlah baris terdampak dan distribusi selisih tanggal:
- **Jenis anomali (acak / berpola):** dan alasan klasifikasi kamu:
- Hipotesis penyebab (migration error? clock skew? fraud?):
- Apakah anomali ini berkorelasi dengan Default? Implikasi untuk analisis risiko:
- Keputusan penanganan (drop / flag / pertahankan) dan alasan:

> 

In [ ]:
# TODO: Implementasi penanganan anomali tanggal


---
### 2.7 Penanganan Business Logic Error: Transaksi PayLater Melebihi Limit

**Ini adalah anomali paling kritis di dataset ini.** Lebih dari 50% transaksi GoPayLater memiliki `amount` yang melebihi `paylater_limit` user, termasuk user Basic tier yang seharusnya tidak punya PayLater sama sekali.

| Tipe Pelanggaran | Deskripsi |
|---|---|
| **Basic tier pakai PayLater** | User dengan `paylater_limit = 0` bertransaksi dengan GoPayLater |
| **Plus tier melebihi limit** | User PayLater sah, tapi `amount > paylater_limit` |
| **Transaksi valid** | User Plus dengan `amount ≤ paylater_limit` |

> 🧠 **Critical Thinking Prompt:**  
> Jangan drop transaksi over-limit — ini adalah **data paling berharga** untuk memahami bug dan pola default.  
> Pertahankan dengan flag, lalu analisis secara terpisah.  
> **Dropping = menghilangkan bukti.**

In [ ]:
# Merge transaksi GoPayLater dengan data paylater_limit user
# Identifikasi tipe pelanggaran untuk setiap transaksi

df27 = df_users.merge(df_trx,
on="user_id", how="left")

df27

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status
0,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0003439,SVC-001,2023-08-31 17:00:00,GoPay,704068.00,0.00,Paid
1,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0137991,SVC-020,2023-08-15 06:00:00,GO-PAY,195808.00,0.00,Paid
2,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0172755,SVC-013,2023-10-08 01:00:00,gopay,606859.00,0.00,Paid
3,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0259404,SVC-019,2023-03-22 00:00:00,GoPay,356899.00,0.00,Paid
4,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0282350,SVC-007,2023-02-08 21:00:00,GoPayLater,186928.00,0.00,Paid
...,...,...,...,...,...,...,...,...,...,...,...,...
300003,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0188082,SVC-019,2023-09-04 03:00:00,GoPay,236650.00,0.00,Paid
300004,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0194534,SVC-017,2023-07-06 19:00:00,gopay,138743.00,0.00,Paid
300005,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0199113,SVC-004,2023-08-06 15:00:00,GoPayLater,749400.00,0.00,Paid
300006,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0232900,SVC-008,2023-02-28 21:00:00,gopay,568217.00,0.00,Paid


In [32]:
df27["paylater_limit"].value_counts()

paylater_limit
0          119144
500000      72234
1500000     54281
3000000     36791
5000000     17558
Name: count, dtype: int64

In [54]:
# Kuantifikasi: berapa jumlah dan total nilai (Rupiah) dari setiap tipe pelanggaran?
total_exceedPayLater = df27.loc[df27["paylater_limit"] > df27["amount"], "amount"].sum()
total_exceedPayLater

np.float64(54937772859.0)

In [55]:
total_zeroPayLater = df27.loc[df27["paylater_limit"] == 0, "amount"].sum()
total_zeroPayLater

np.float64(49663530665.0)

In [37]:
total_ZeroExceed = df27.loc[
    (df27["paylater_limit"] == 0) & (df27["amount"] > df27["paylater_limit"]), "amount"
].sum()
total_ZeroExceed

np.float64(49663530665.0)

In [58]:
#Flagging

def flag(row):
    if (row["paylater_limit"] == 0) & (row["amount"] > 0):
        return "unauthorized_paylater"
    elif row["amount"] > row["paylater_limit"]:
        return "over_limit_paylater"
    elif (row["paylater_limit"] > row["amount"]):
        return "normal"

df27["flag"] = df27.apply(flag, axis=1)

In [41]:
df27

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,flag
0,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0003439,SVC-001,2023-08-31 17:00:00,GoPay,704068.00,0.00,Paid,unauthorized_paylater
1,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0137991,SVC-020,2023-08-15 06:00:00,GO-PAY,195808.00,0.00,Paid,unauthorized_paylater
2,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0172755,SVC-013,2023-10-08 01:00:00,gopay,606859.00,0.00,Paid,unauthorized_paylater
3,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0259404,SVC-019,2023-03-22 00:00:00,GoPay,356899.00,0.00,Paid,unauthorized_paylater
4,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0282350,SVC-007,2023-02-08 21:00:00,GoPayLater,186928.00,0.00,Paid,unauthorized_paylater
...,...,...,...,...,...,...,...,...,...,...,...,...,...
300003,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0188082,SVC-019,2023-09-04 03:00:00,GoPay,236650.00,0.00,Paid,unauthorized_paylater
300004,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0194534,SVC-017,2023-07-06 19:00:00,gopay,138743.00,0.00,Paid,unauthorized_paylater
300005,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0199113,SVC-004,2023-08-06 15:00:00,GoPayLater,749400.00,0.00,Paid,unauthorized_paylater
300006,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0232900,SVC-008,2023-02-28 21:00:00,gopay,568217.00,0.00,Paid,unauthorized_paylater


In [59]:
df27.loc[df27["flag"] == "normal", :]

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,flag
6,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0017474,SVC-010,2023-06-29 12:00:00,gopay,200782.00,0.00,Paid,normal
7,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0085613,SVC-007,2023-08-14 06:00:00,GoPay,251279.00,0.00,Paid,normal
8,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0089307,SVC-008,2023-01-24 20:00:00,GoPay,191460.00,0.00,Paid,normal
9,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0101411,SVC-003,2023-05-24 07:00:00,gopay,88206.00,0.00,Paid,normal
10,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0103354,SVC-007,2023-05-18 08:00:00,GoPayLater,357436.00,37815.00,Default,normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...
299982,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0165566,SVC-016,2023-03-17 16:00:00,GoPay,539583.00,0.00,Paid,normal
299983,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0187097,SVC-018,2023-05-06 13:00:00,gopay,708976.00,0.00,Paid,normal
299984,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0209452,SVC-014,2023-02-28 07:00:00,GoPay,686240.00,0.00,Paid,normal
299985,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0211865,SVC-003,2023-11-24 13:00:00,GoPayLater,172077.00,13120.00,Default,normal


In [60]:
df27.loc[df27["flag"].isnull(), :]

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,flag
12962,GP-001524,2021-01-27,Plus,472.00,500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75508,GP-008804,2021-05-11,Basic,492.00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
160580,GP-018739,2021-10-08,Plus,745.00,500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
201812,GP-023538,2021-04-07,Plus,NaN,500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
209201,GP-024403,2021-03-07,Basic,323.00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
228993,GP-026708,2021-06-13,Basic,406.00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
251878,GP-029370,2022-06-23,Basic,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
279902,GP-032647,2022-05-22,Plus,438.00,500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [46]:
(df27["amount"] == 0).sum()

np.int64(0)

In [ ]:
# Kritis: apakah transaksi over-limit berkorelasi dengan payment_status = Default?
# Bandingkan default rate antara: transaksi valid vs over-limit


**✍️ Analisis & Justifikasi:**
- Jumlah dan persentase setiap tipe pelanggaran:
- Total nilai Rupiah yang terlibat dalam pelanggaran:
- Apakah over-limit berkorelasi dengan Default? Temuan kamu:
- Keputusan penanganan (flag, bukan drop) dan kolom flag yang kamu buat:
- Hipotesis mengapa bug ini bisa terjadi di sistem:

> 

In [ ]:
# TODO: Buat kolom flag untuk tipe pelanggaran PayLater
# Contoh: 'valid', 'over_limit', 'unauthorized'


---
### 2.8 Penanganan Duplikat & Integritas Data

In [ ]:
# 1. Cek exact duplicates di setiap tabel


In [ ]:
# 2. Cek duplikat trx_id


In [ ]:
# 3. Cek service_id di transactions yang tidak ada di services


In [ ]:
# 4. Cek inkonsistensi logika: payment_status = Pending tapi late_fee > 0


**✍️ Analisis & Justifikasi:**
- Masalah yang ditemukan dan jumlah baris terdampak:
- Hipotesis untuk setiap masalah:
- Keputusan penanganan per masalah:

> 

In [ ]:
# TODO: Implementasi keputusan penanganan masalah integritas


---
## 3. Feature Engineering

### 3.1 Fitur Wajib

Buat 8 kolom berikut. Sertakan penjelasan singkat business value-nya di setiap fitur.

#### ⚙️ `payment_method_clean`
*(Sudah dibuat di Section 2.3 — pastikan sudah ada di df_trx)*

#### ⚙️ `credit_score_tier`

> 💡 Default threshold: Poor (300–499), Fair (500–649), Good (650–749), Excellent (750–850).  
> Sesuaikan jika analisis distribusi kamu menunjukkan pembagian yang lebih bermakna secara bisnis.

**✍️ Threshold yang kamu pilih dan alasannya:**

> 

In [ ]:
# TODO: Buat credit_score_tier di df_users
# Pertimbangkan: bagaimana menangani user yang credit_score-nya NaN?


#### ⚙️ `is_paylater_violation`

> 💡 Perlu merge df_trx dengan df_users untuk mendapatkan paylater_limit per transaksi.

In [ ]:
# TODO: Buat is_paylater_violation (boolean)
# True jika payment_method_clean == 'gopaylater' AND amount > paylater_limit


#### ⚙️ `paylater_usage_ratio`

> 💡 Hanya relevan untuk transaksi GoPayLater. Untuk transaksi non-PayLater, isi dengan NaN.

In [ ]:
# TODO: Buat paylater_usage_ratio (amount / paylater_limit)
# Handle division by zero untuk user dengan paylater_limit = 0


#### ⚙️ `user_tenure_days`

> 💡 Tentukan sendiri tanggal referensi yang kamu gunakan dan justifikasikan.

**✍️ Tanggal referensi yang kamu gunakan dan alasannya:**

> 

In [ ]:
# TODO: Buat user_tenure_days di df_users


#### ⚙️ `has_late_fee`

In [ ]:
# TODO: Buat has_late_fee (boolean: True jika late_fee > 0)
# Pastikan menggunakan late_fee yang sudah di-clean dari Section 2.5


#### ⚙️ `is_default`

In [ ]:
# TODO: Buat is_default (boolean: True jika payment_status == 'Default')


#### ⚙️ `service_category`

> 💡 Join df_trx dengan df_services untuk mendapatkan kategori layanan per transaksi.

In [ ]:
# TODO: Buat service_category dengan merge ke df_services


---
### 3.2 Fitur Pilihan (Minimal 2)

Pilih minimal 2 dari: `default_rate_per_user`, `avg_amount_per_service_category`, `is_high_risk_transaction`, `credit_utilization_band`, atau fitur buatan sendiri.

#### ⚙️ Fitur Pilihan 1: [Isi nama fitur]

**✍️ Business value dari fitur ini:**

> 

In [ ]:
# TODO: Implementasi Fitur Pilihan 1


#### ⚙️ Fitur Pilihan 2: [Isi nama fitur]

**✍️ Business value dari fitur ini:**

> 

In [ ]:
# TODO: Implementasi Fitur Pilihan 2


---
## 4. Exploratory Data Analysis

> **Aturan:** Semua analisis menggunakan pandas — tanpa visualisasi.  
> Gunakan `.groupby()`, `.agg()`, `.value_counts()`, filtering, sorting, dan **merge antar tabel** saat dibutuhkan.  
> Setiap jawaban **wajib disertai insight** di markdown cell yang tersedia.

---
### 4.1 Analisis Transaksi & Metode Pembayaran

**Soal 1:** Berapa total GTV (Gross Transaction Value) keseluruhan? Breakdown GTV per `payment_method_clean`. Metode mana yang paling dominan dan apa implikasi bisnisnya?

In [ ]:
# Soal 1


**✍️ Insight:**

> 

**Soal 2:** Berapa distribusi `payment_status` secara keseluruhan? Kemudian breakdown **default rate** per `payment_method_clean`. Apakah GoPayLater memiliki default rate yang lebih tinggi?

In [ ]:
# Soal 2


**✍️ Insight:**

> 

**Soal 3:** Berapa rata-rata, median, dan standar deviasi `amount` per `payment_method_clean`? Apa yang bisa disimpulkan dari perbedaan mean vs median?

In [ ]:
# Soal 3


**✍️ Insight:**

> 

**Soal 4:** Analisis `late_fee`: Berapa persentase transaksi yang dikenakan denda? Berapa total `late_fee` yang terkumpul? Breakdown per `payment_method_clean`.

In [ ]:
# Soal 4


**✍️ Insight:**

> 

---
### 4.2 Analisis Risiko Kredit & Profil User

**Soal 5:** Berapa distribusi `credit_score_tier`? Kemudian bandingkan **default rate** (untuk transaksi GoPayLater) antar `credit_score_tier`. Apakah user dengan credit score rendah memiliki default rate yang lebih tinggi?

In [ ]:
# Soal 5
# Hint: merge df_trx (filter GoPayLater) dengan df_users, lalu groupby credit_score_tier


**✍️ Insight:**

> 

**Soal 6:** Berapa persentase `is_paylater_violation = True`? Breakdown antara: Basic tier pakai PayLater vs Plus tier melebihi limit. Berapa total nilai Rupiah yang terlibat?

In [ ]:
# Soal 6


**✍️ Insight:**

> 

**Soal 7:** Apakah ada korelasi antara `paylater_usage_ratio` dan `is_default`? Bandingkan rata-rata `paylater_usage_ratio` antara transaksi yang Default vs yang tidak.

In [ ]:
# Soal 7


**✍️ Insight:**

> 

**Soal 8:** Berapa distribusi `gopay_tier` di antara user yang pernah Default? Apakah user Basic yang 'membobol' sistem PayLater memiliki default rate lebih tinggi dari user Plus yang sah?

In [ ]:
# Soal 8


**✍️ Insight:**

> 

---
### 4.3 Analisis Layanan & Kategori

**Soal 9:** Berapa total GTV dan jumlah transaksi per `service_category`? Kategori mana yang paling tinggi volumenya?

In [ ]:
# Soal 9


**✍️ Insight:**

> 

**Soal 10:** Berapa **default rate** per `service_category` untuk transaksi GoPayLater? Layanan mana yang paling berisiko untuk dibayar dengan PayLater?

In [ ]:
# Soal 10


**✍️ Insight:**

> 

**Soal 11:** Top 5 `service_name` berdasarkan total `late_fee` yang dikumpulkan. Apakah ini mengindikasikan layanan tertentu lebih sering mengalami keterlambatan pembayaran?

In [ ]:
# Soal 11


**✍️ Insight:**

> 

---
### 4.4 Analisis Sistem & Deteksi Anomali *(Implicit — Business Sense Required)*

> Kamu diminta tim **Risk & Compliance** untuk menyusun laporan investigasi sistem PayLater.  
> Temuan ini akan digunakan untuk: (a) menentukan apakah PayLater perlu di-suspend sementara,  
> (b) mengidentifikasi user yang perlu limit adjustment, dan  
> (c) mengestimasi **total kerugian potensial** dari bug yang ada.

Pilih minimal **2 angle analisis** yang paling relevan untuk menjawab kebutuhan investigasi tersebut.

#### 🔍 Investigasi — Angle 1: [Isi judul]

**✍️ Mengapa kamu memilih angle ini untuk investigasi sistem?**

> 

In [ ]:
# Angle 1


**✍️ Insight & Rekomendasi untuk Tim Risk & Compliance:**

> 

#### 🔍 Investigasi — Angle 2: [Isi judul]

**✍️ Mengapa kamu memilih angle ini?**

> 

In [ ]:
# Angle 2


**✍️ Insight & Rekomendasi:**

> 

---
### 4.5 Credit Risk Profiling *(Implicit — Open Ended)*

> Kamu diminta **Chief Risk Officer GoPay** untuk menyusun rekomendasi perbaikan algoritma credit scoring.  
> Tujuan: menentukan kriteria yang lebih ketat untuk pemberian limit PayLater,  
> sehingga NPL bisa ditekan tanpa terlalu banyak membatasi user yang sebenarnya *creditworthy*.

> 🧠 **Critical Thinking Prompt:**  
> Apakah user dengan credit score rendah **selalu** berisiko?  
> Bagaimana dengan user baru yang belum punya credit score sama sekali?  
> Temukan **sweet spot** antara risk mitigation dan business growth.

**Ekspektasi minimal:**
- Minimal 3 variabel/fitur berbeda yang kamu identifikasi sebagai prediktor default yang signifikan
- Profil 'high-risk user' berdasarkan kombinasi variabel tersebut
- Minimal 1 rekomendasi konkret untuk kebijakan limit PayLater yang berbasis data

**✍️ Definisi 'high-risk user' menurut kamu (dalam konteks kredit GoPay):**

> 

#### 📊 Prediktor Default 1: [Nama Variabel]

In [ ]:
# Prediktor 1


#### 📊 Prediktor Default 2: [Nama Variabel]

In [ ]:
# Prediktor 2


#### 📊 Prediktor Default 3: [Nama Variabel]

In [ ]:
# Prediktor 3


#### 🎯 Profil High-Risk User vs Average User

In [ ]:
# Bandingkan karakteristik high-risk user vs keseluruhan user PayLater


**✍️ Rekomendasi Kebijakan Limit PayLater untuk Chief Risk Officer:**

> 

---
## 5. Export Clean Dataset

In [ ]:
# Gabungkan ketiga tabel menjadi satu dataframe final
# Gunakan LEFT JOIN dengan df_trx sebagai tabel utama
# Sertakan semua fitur baru yang telah dibuat

# TODO: Implementasi JOIN
# df_final = df_trx.merge(df_users[...], on='user_id', how='left')
#                  .merge(df_services[...], on='service_id', how='left')

# Export
# df_final.to_csv('gopay_clean.csv', index=False)
# print(f'Dataset berhasil disimpan: gopay_clean.csv')
# print(f'Shape final: {df_final.shape}')
# print(f'Kolom baru yang ditambahkan: {[c for c in df_final.columns if c not in df_trx_raw.columns]}')


---
## 6. Ringkasan & Refleksi

**Keputusan Data Cleaning yang paling challenging dan mengapa:**

> 

**Temuan paling menarik dari EDA (khususnya terkait risiko kredit):**

> 

**Rekomendasi bisnis utama yang bisa diberikan kepada tim Risk Management GoPay:**

> 